# Plot saved polygon shapes on a map
Load JSON/GeoJSON files created by **draw_polygon.py** and display all shapes on a Cartopy coastline map.

1. Install the dependencies if needed.
2. Edit `INPUTS` below to select files or a folder.
3. Run the remaining cells in order.

Coordinates must be **[longitude, latitude]**, in WGS84 degrees. Polygon, MultiPolygon, Feature, and FeatureCollection objects are supported; polygon holes are preserved. Natural Earth map data downloads and is cached on first use.

In [ ]:
# Uncomment and run if these packages are missing from this notebook kernel:
# %pip install matplotlib cartopy shapely

In [ ]:
from pathlib import Path
import glob
import json
import warnings
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import shape
from shapely.validation import explain_validity

# EDIT THESE SETTINGS. Relative paths are relative to the notebook working directory.
INPUTS = ["polygon1.json"]
# INPUTS = ["polygon1.json", "polygon2.json"]
# INPUTS = ["./shapes"]                 # all .json/.geojson files in a folder
# INPUTS = ["./shapes/*.json"]

MAP_RESOLUTION = "50m"                 # "110m" (fast), "50m", or "10m" (detailed)
EXTENT = None                         # Auto-fit; or [west, east, south, north]
# EXTENT = [-5, 10, 48, 60]             # North Sea region
SHOW_VERTICES = True
SHOW_VERTEX_NUMBERS = False
FILL_ALPHA = 0.20
OUTPUT = None                         # Or "shapes_map.png" / "shapes_map.pdf"

print("Working directory:", Path.cwd())

In [ ]:
def resolve_files(inputs):
    files = set()
    for entry in inputs:
        matches = glob.glob(str(Path(entry).expanduser()))
        if not matches:
            raise FileNotFoundError(f"No files match {entry!r}. Edit INPUTS above.")
        for match in matches:
            path = Path(match)
            if path.is_dir():
                files.update(p.resolve() for p in path.iterdir()
                             if p.is_file() and p.suffix.lower() in (".json", ".geojson"))
            elif path.is_file():
                files.add(path.resolve())
    if not files:
        raise FileNotFoundError("No JSON/GeoJSON files found.")
    return sorted(files)


def extract_polygons(obj, label):
    """Return (label, Shapely geometry) pairs, retaining feature names."""
    if not isinstance(obj, dict):
        raise ValueError(f"{label}: expected a GeoJSON object.")
    kind = obj.get("type")
    if kind == "FeatureCollection":
        results = []
        for index, feature in enumerate(obj.get("features", []), 1):
            results.extend(extract_polygons(feature, f"{label} · {index}"))
        return results
    if kind == "Feature":
        properties = obj.get("properties") or {}
        name = properties.get("name") or properties.get("title")
        return extract_polygons(obj.get("geometry"), f"{label} · {name}" if name else label)
    if kind not in ("Polygon", "MultiPolygon"):
        raise ValueError(f"{label}: unsupported geometry {kind!r}; expected polygons.")
    geometry = shape(obj)
    if geometry.is_empty or not geometry.is_valid:
        raise ValueError(f"{label}: empty or invalid polygon ({explain_validity(geometry)}).")
    parts = [geometry] if kind == "Polygon" else geometry.geoms
    for polygon in parts:
        for ring in [polygon.exterior, *polygon.interiors]:
            points = list(ring.coords)
            if any(not (-180 <= p[0] <= 180 and -90 <= p[1] <= 90) for p in points):
                raise ValueError(f"{label}: coordinates must be longitude/latitude degrees.")
            if any(abs(a[0] - b[0]) > 180 for a, b in zip(points, points[1:])):
                raise ValueError(f"{label}: polygons crossing the 180-degree seam are unsupported.")
    return [(label, geometry)]


files = resolve_files(INPUTS)
shapes = []
for file in files:
    try:
        with file.open(encoding="utf-8-sig") as stream:
            shapes.extend(extract_polygons(json.load(stream), file.stem))
    except (ValueError, TypeError, KeyError) as exc:
        raise ValueError(f"Cannot load {file}: {exc}") from exc
if not shapes:
    raise ValueError("The selected files contain no polygons.")
print(f"Loaded {len(shapes)} shape(s) from {len(files)} file(s).")
for label, geometry in shapes:
    print(f"  {label}: {geometry.geom_type}, bounds={geometry.bounds}")

In [ ]:
if MAP_RESOLUTION not in ("110m", "50m", "10m"):
    raise ValueError("MAP_RESOLUTION must be '110m', '50m', or '10m'.")
if EXTENT is None:
    bounds = [geometry.bounds for _, geometry in shapes]
    west, south = min(b[0] for b in bounds), min(b[1] for b in bounds)
    east, north = max(b[2] for b in bounds), max(b[3] for b in bounds)
    dx, dy = max(0.1, (east-west)*0.12), max(0.1, (north-south)*0.12)
    extent = [max(-180, west-dx), min(180, east+dx),
              max(-90, south-dy), min(90, north+dy)]
else:
    extent = list(EXTENT)
if len(extent) != 4:
    raise ValueError("EXTENT must contain west, east, south, north.")
w, e, s, n = extent
if not (-180 <= w < e <= 180 and -90 <= s < n <= 90):
    raise ValueError("Invalid map extent.")

projection = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(13, 9), subplot_kw={"projection": projection})
ax.set_global()
ax.set_facecolor("#dceef5")
ax.add_feature(cfeature.LAND.with_scale(MAP_RESOLUTION), facecolor="#eee9dc", zorder=0)
ax.coastlines(resolution=MAP_RESOLUTION, linewidth=0.7, color="#555555", zorder=1)
ax.add_feature(cfeature.BORDERS.with_scale(MAP_RESOLUTION), linewidth=0.4, edgecolor="#999999", zorder=1)
grid = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4)
grid.top_labels = grid.right_labels = False

handles = []
colors = plt.get_cmap("tab20")
for index, (label, geometry) in enumerate(shapes):
    color = colors(index % 20)
    ax.add_geometries([geometry], crs=projection, facecolor=color,
                      edgecolor="none", alpha=FILL_ALPHA, zorder=2)
    ax.add_geometries([geometry], crs=projection, facecolor="none",
                      edgecolor=color, linewidth=2, zorder=3)
    handles.append(Patch(facecolor=color, edgecolor=color, label=label))
    polygons = [geometry] if geometry.geom_type == "Polygon" else geometry.geoms
    for part_index, polygon in enumerate(polygons, 1):
        for ring_index, ring in enumerate([polygon.exterior, *polygon.interiors]):
            points = list(ring.coords)[:-1]  # skip duplicate closure vertex
            if SHOW_VERTICES:
                ax.scatter([p[0] for p in points], [p[1] for p in points],
                           s=18, color=[color], transform=projection, zorder=4)
            if SHOW_VERTEX_NUMBERS:
                for vertex_index, point in enumerate(points, 1):
                    ax.annotate(f"{part_index}.{ring_index}.{vertex_index}", xy=point[:2],
                                xytext=(4, 4), textcoords="offset points", fontsize=8, zorder=5)

ax.set_title(f"Saved polygons · {len(shapes)} shape(s)", fontsize=15, pad=20)
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.04, 1), title="Shapes", fontsize=9)
fig.tight_layout()
if OUTPUT:
    destination = Path(OUTPUT).expanduser().resolve()
    if destination.suffix.lower() not in (".png", ".pdf", ".svg"):
        raise ValueError("OUTPUT must end in .png, .pdf, or .svg.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(destination, dpi=180, bbox_inches="tight")
    print("Saved:", destination)
plt.show()

### Notes
- The map automatically fits all loaded shapes. Set `EXTENT` to keep a fixed view.
- For interactive toolbar pan/zoom inside Jupyter, install `ipympl` and run `%matplotlib widget` **before** the plotting cell. The normal inline view works without it.
- Vertex labels use `polygon.ring.vertex` numbering; ring 0 is the exterior.
- Files are validated, not silently repaired. Input files are never modified.
- This notebook displays geographic outlines, not AIS tracks.